# 3. Organoid–Cell Relationship

## Purpose
This notebook assigns each single cell and nucleocentric object to its parent organoid,
then computes spatial relationship features (Euclidean distance, Mahalanobis distance,
and shell classification) for each cell relative to its parent organoid centroid.

This is **step 3 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
Three parquet files from `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:
- `sc_profiles_{well_fov}.parquet` — merged Nuclei + Cell + Cytoplasm features
- `organoid_profiles_{well_fov}.parquet` — organoid features
- `nucleocentric_profiles_{well_fov}.parquet` — nucleocentric features

## Outputs
Three enriched parquet files written to `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`:

| File | Added columns |
|---|---|
| `sc_profiles_{well_fov}_related.parquet` | `ParentOrganoid`, shell/distance features |
| `organoid_profiles_{well_fov}_related.parquet` | `OrganoidSingleCellCount` |
| `nucleocentric_profiles_{well_fov}_related.parquet` | `ParentOrganoid` |

## Notes
- Parent organoid assignment uses bbox containment: a cell is assigned to the first
  organoid whose bounding box contains the cell's nuclear centroid.
- Spatial features are computed using Mahalanobis distance with a regularized covariance
  matrix (applied automatically when cell count is low).
- Shell classification divides cells into 4 concentric shells from organoid centroid
  outward, requiring at least 3 cells per shell.

In [1]:
import os
import pathlib

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C10-2"
    image_based_profiles_subparent_name = "image_based_profiles"

### Pathing

In [3]:
# input paths
sc_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve(strict=True)
# output paths
sc_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
sc_profile_df = pd.read_parquet(sc_profile_path)
nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
organoid_profile_df = pd.read_parquet(organoid_profile_path)
print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")

Single-cell profile shape: (11, 11883)
Nucleocentric profile shape: (11, 3074)
Organoid profile shape: (3, 3961)


In [5]:
nucleocentric_df

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature90,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99
0,257,C10-2,-0.003565,-0.221765,0.128936,0.045298,-0.044059,0.301420,0.124469,-0.107604,...,-1.669074,5.728443,-9.407360,-2.038582,1.370812,-5.477564,-0.336243,1.502100,1.546016,0.210300
1,514,C10-2,-0.448317,-0.227693,0.127213,-0.005069,0.028333,0.112086,0.028173,-0.080888,...,-0.845948,1.572523,-6.513134,-3.327943,5.796319,-3.204541,-0.909036,3.659749,-3.007935,1.133443
2,1028,C10-2,-0.372906,-0.141308,0.181552,-0.005005,0.046912,0.081046,0.008974,-0.084817,...,-2.569808,2.079188,-8.914544,-0.971781,3.080546,-6.735629,-1.390779,2.282022,1.280657,-0.801634
3,1285,C10-2,-0.173261,-0.330922,0.239251,0.008114,-0.084119,0.230205,0.123472,-0.168975,...,-2.485241,3.528395,-4.063998,-3.547382,2.289299,-2.786486,0.148371,3.511132,-2.909451,0.473595
4,1542,C10-2,-0.477709,-0.282062,0.235665,-0.032700,-0.003505,0.119395,0.032428,-0.049148,...,-0.873972,-2.390701,-4.856587,-3.907551,6.513190,3.495226,-0.992177,3.060692,-1.848124,-4.039866
5,1799,C10-2,-0.217139,-0.220789,0.223406,-0.012517,-0.120618,0.218664,0.079021,-0.116704,...,1.035481,0.765297,-3.166644,-3.280701,-0.792202,4.873670,4.745506,2.032445,0.774616,-1.587440
6,2056,C10-2,-0.456463,-0.317283,0.158936,-0.024655,-0.192885,0.236079,0.041396,-0.176195,...,-1.963491,3.423923,-2.937624,-0.038833,1.354741,-2.227046,-2.480601,1.697044,-3.202453,-1.812880
7,2313,C10-2,-0.015282,-0.131564,0.017068,-0.168012,-0.099932,0.401822,0.031163,-0.179202,...,1.732337,0.673396,-11.011210,0.761109,7.344636,2.937018,2.961241,-0.441801,-1.872351,-2.308038
8,2570,C10-2,-0.043089,-0.235939,0.001961,-0.041636,-0.141147,0.344937,0.144757,-0.177636,...,-2.654626,2.681708,-4.684893,-0.672743,2.141154,-2.931852,-3.241089,3.328008,-1.768050,-1.157220
9,2827,C10-2,-0.481294,-0.185675,0.218604,-0.033112,-0.031565,0.019963,-0.085351,0.016684,...,-1.035921,4.811902,-4.739573,-1.540166,3.403223,-0.343314,-2.886456,3.433978,-1.822761,1.193305


In [6]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    if "area" in x.lower() and "center" in x.lower() and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_AreaSizeShape_CenterX',
 'Nuclei_NoChannel_AreaSizeShape_CenterY',
 'Nuclei_NoChannel_AreaSizeShape_CenterZ']

In [7]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

# When sorted alphabetically, the bbox column names fall in this order:
#   [0] = *MaxX, [1] = *MaxY, [2] = *MaxZ, [3] = *MinX, [4] = *MinY, [5] = *MinZ
# This ordering is assumed in the bbox tuple construction below.

In [8]:
# Initialize ParentOrganoid to -1 (sentinel for unassigned cells).
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array

# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Build the bbox tuple using the sorted column order documented in the cell above:
    # sorted alphabetically gives [MaxX, MaxY, MaxZ, MinX, MinY, MinZ]
    # so indices [5]=MinZ, [4]=MinY, [3]=MinX, [2]=MaxZ, [1]=MaxY, [0]=MaxX.
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # First-match-wins: if organoid bboxes overlap, a cell is assigned to the first
    # organoid whose bbox contains it and is never reassigned to a later one.
    # Both masks are NumPy arrays (positional) to avoid pandas index misalignment.
    unassigned_mask = sc_profile_df["ParentOrganoid"].values == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[sc_profile_df.index[final_mask], "ParentOrganoid"] = organoid_row[
        "object_id"
    ]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 3/3 [00:00<00:00, 1622.97it/s]

Assigned 9 cells to organoids
Unassigned cells: 2


### Add single-cell counts for each organoid

In [9]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="OrganoidSingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("OrganoidSingleCellCount")
organoid_profile_df.insert(2, "OrganoidSingleCellCount", sc_count)

### Empty dataframe fallbacks

If either the organoid or SC profile is empty for this well-FOV, a placeholder row
is inserted so that downstream merges always find consistent columns.

In [10]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["OrganoidSingleCellCount"] = (
    organoid_profile_df["OrganoidSingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C10-2,4,1210841.0,838.615864,429.637973,9.794105,1825596.0,684,998,...,1152.351969,1105.363718,1147.235766,1103.937977,1143.647327,1149.664104,1149.646407,1102.551268,1144.905563,1151.809001
1,2,C10-2,5,749790.0,304.417777,879.567732,9.106898,1071408.0,207,415,...,303.886884,303.366418,304.176987,302.987602,303.820368,303.924470,303.913711,303.237717,303.886440,303.678211
2,3,C10-2,0,15506.0,987.558365,1484.026699,1.500000,19980.0,934,1045,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
if organoid_profile_df.empty:
    # Write the empty DataFrame as-is. Parquet preserves schema (columns) even with
    # zero rows, so downstream union_by_name in 5.combining_profiles handles this
    # correctly. A fake zero-filled row was previously inserted here but produced
    # object_id=0 (the background label), creating a spurious organoid row that
    # propagated through all downstream stages.
    pass

In [12]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (11, 11884)


In [13]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [14]:
nucleocentric_df

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature90,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99
0,257,C10-2,-0.003565,-0.221765,0.128936,0.045298,-0.044059,0.301420,0.124469,-0.107604,...,-1.669074,5.728443,-9.407360,-2.038582,1.370812,-5.477564,-0.336243,1.502100,1.546016,0.210300
1,514,C10-2,-0.448317,-0.227693,0.127213,-0.005069,0.028333,0.112086,0.028173,-0.080888,...,-0.845948,1.572523,-6.513134,-3.327943,5.796319,-3.204541,-0.909036,3.659749,-3.007935,1.133443
2,1028,C10-2,-0.372906,-0.141308,0.181552,-0.005005,0.046912,0.081046,0.008974,-0.084817,...,-2.569808,2.079188,-8.914544,-0.971781,3.080546,-6.735629,-1.390779,2.282022,1.280657,-0.801634
3,1285,C10-2,-0.173261,-0.330922,0.239251,0.008114,-0.084119,0.230205,0.123472,-0.168975,...,-2.485241,3.528395,-4.063998,-3.547382,2.289299,-2.786486,0.148371,3.511132,-2.909451,0.473595
4,1542,C10-2,-0.477709,-0.282062,0.235665,-0.032700,-0.003505,0.119395,0.032428,-0.049148,...,-0.873972,-2.390701,-4.856587,-3.907551,6.513190,3.495226,-0.992177,3.060692,-1.848124,-4.039866
5,1799,C10-2,-0.217139,-0.220789,0.223406,-0.012517,-0.120618,0.218664,0.079021,-0.116704,...,1.035481,0.765297,-3.166644,-3.280701,-0.792202,4.873670,4.745506,2.032445,0.774616,-1.587440
6,2056,C10-2,-0.456463,-0.317283,0.158936,-0.024655,-0.192885,0.236079,0.041396,-0.176195,...,-1.963491,3.423923,-2.937624,-0.038833,1.354741,-2.227046,-2.480601,1.697044,-3.202453,-1.812880
7,2313,C10-2,-0.015282,-0.131564,0.017068,-0.168012,-0.099932,0.401822,0.031163,-0.179202,...,1.732337,0.673396,-11.011210,0.761109,7.344636,2.937018,2.961241,-0.441801,-1.872351,-2.308038
8,2570,C10-2,-0.043089,-0.235939,0.001961,-0.041636,-0.141147,0.344937,0.144757,-0.177636,...,-2.654626,2.681708,-4.684893,-0.672743,2.141154,-2.931852,-3.241089,3.328008,-1.768050,-1.157220
9,2827,C10-2,-0.481294,-0.185675,0.218604,-0.033112,-0.031565,0.019963,-0.085351,0.016684,...,-1.035921,4.811902,-4.739573,-1.540166,3.403223,-0.343314,-2.886456,3.433978,-1.822761,1.193305


In [15]:
# Propagate ParentOrganoid to nucleocentric profiles.
# Nucleocentric objects share object_id with their parent nucleus, so joining
# on object_id + image_set carries the organoid assignment through.
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

In [16]:
nucleocentric_df

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,257,C10-2,-0.003565,-0.221765,0.128936,0.045298,-0.044059,0.301420,0.124469,-0.107604,...,5.728443,-9.407360,-2.038582,1.370812,-5.477564,-0.336243,1.502100,1.546016,0.210300,2
1,514,C10-2,-0.448317,-0.227693,0.127213,-0.005069,0.028333,0.112086,0.028173,-0.080888,...,1.572523,-6.513134,-3.327943,5.796319,-3.204541,-0.909036,3.659749,-3.007935,1.133443,2
2,1028,C10-2,-0.372906,-0.141308,0.181552,-0.005005,0.046912,0.081046,0.008974,-0.084817,...,2.079188,-8.914544,-0.971781,3.080546,-6.735629,-1.390779,2.282022,1.280657,-0.801634,2
3,1285,C10-2,-0.173261,-0.330922,0.239251,0.008114,-0.084119,0.230205,0.123472,-0.168975,...,3.528395,-4.063998,-3.547382,2.289299,-2.786486,0.148371,3.511132,-2.909451,0.473595,2
4,1542,C10-2,-0.477709,-0.282062,0.235665,-0.032700,-0.003505,0.119395,0.032428,-0.049148,...,-2.390701,-4.856587,-3.907551,6.513190,3.495226,-0.992177,3.060692,-1.848124,-4.039866,-1
5,1799,C10-2,-0.217139,-0.220789,0.223406,-0.012517,-0.120618,0.218664,0.079021,-0.116704,...,0.765297,-3.166644,-3.280701,-0.792202,4.873670,4.745506,2.032445,0.774616,-1.587440,-1
6,2056,C10-2,-0.456463,-0.317283,0.158936,-0.024655,-0.192885,0.236079,0.041396,-0.176195,...,3.423923,-2.937624,-0.038833,1.354741,-2.227046,-2.480601,1.697044,-3.202453,-1.812880,1
7,2313,C10-2,-0.015282,-0.131564,0.017068,-0.168012,-0.099932,0.401822,0.031163,-0.179202,...,0.673396,-11.011210,0.761109,7.344636,2.937018,2.961241,-0.441801,-1.872351,-2.308038,1
8,2570,C10-2,-0.043089,-0.235939,0.001961,-0.041636,-0.141147,0.344937,0.144757,-0.177636,...,2.681708,-4.684893,-0.672743,2.141154,-2.931852,-3.241089,3.328008,-1.768050,-1.157220,1
9,2827,C10-2,-0.481294,-0.185675,0.218604,-0.033112,-0.031565,0.019963,-0.085351,0.016684,...,4.811902,-4.739573,-1.540166,3.403223,-0.343314,-2.886456,3.433978,-1.822761,1.193305,1


## Get single cell and organoid relationships and spatial distributions

In [17]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]

In [18]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    # x_y_z_sc_colnames is alphabetically sorted: [CenterX, CenterY, CenterZ]
    # so index [0]=X, [1]=Y, [2]=Z. The dict remaps them to named z/y/x keys.
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

           Reducing to 2 shells for statistical reliability
           Reducing to 2 shells for statistical reliability
No single cells assigned to organoid 3


In [19]:
# Concatenate per-organoid shell results and rename columns to standard feature name format.
# Added columns (under Nuclei_NoChannel_Neighbors_*):
#   ShellAssignments            — shell index (1=innermost, N=outermost) for each cell
#   DistancesFromCenter         — Mahalanobis distance from organoid centroid
#   DistancesFromExterior       — distance from the outermost shell boundary
#   NormalizedDistancesFromCenter — DistancesFromCenter normalized to [0, 1]
#   ShellsUsed                  — total number of shells actually assigned (may be < 4
#                                 if too few cells to fill all shells)
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [20]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [21]:
organoid_profile_df.to_parquet(organoid_profile_output_path, index=False)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C10-2,4,1210841.0,838.615864,429.637973,9.794105,1825596.0,684,998,...,1152.351969,1105.363718,1147.235766,1103.937977,1143.647327,1149.664104,1149.646407,1102.551268,1144.905563,1151.809001
1,2,C10-2,5,749790.0,304.417777,879.567732,9.106898,1071408.0,207,415,...,303.886884,303.366418,304.176987,302.987602,303.820368,303.924470,303.913711,303.237717,303.886440,303.678211
2,3,C10-2,0,15506.0,987.558365,1484.026699,1.500000,19980.0,934,1045,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
sc_profile_with_shells_df.to_parquet(sc_profile_output_path, index=False)
sc_profile_with_shells_df.head()

,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_DNA_Texture_Variance-3-09-256,Cytoplasm_DNA_Texture_Variance-3-10-256,Cytoplasm_DNA_Texture_Variance-3-11-256,Cytoplasm_DNA_Texture_Variance-3-12-256,ParentOrganoid,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,257,C10-2,53529.0,366.587308,869.943171,4.558576,90720.0,330,402,808,...,0.0,0.0,0.0,0.0,2,1.0,63.074321,1.517556,0.976505,2.0
1,514,C10-2,43943.0,314.491660,815.983797,5.786837,75174.0,284,351,770,...,0.0,0.0,0.0,0.0,2,1.0,64.462568,0.129309,0.997998,2.0
2,1028,C10-2,42928.0,241.827199,863.996436,5.076756,67338.0,214,272,800,...,0.0,0.0,0.0,0.0,2,1.0,64.624204,-0.032327,1.000500,2.0
3,1285,C10-2,25653.0,276.793552,930.398979,2.837212,40180.0,239,321,882,...,0.0,0.0,0.0,0.0,2,1.0,58.191257,6.400620,0.900907,2.0
4,1542,C10-2,9672.0,577.095327,934.242349,1.500000,15106.0,532,615,886,...,0.0,0.0,0.0,0.0,-1,NaN,NaN,NaN,NaN,NaN


In [23]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,257,C10-2,-0.003565,-0.221765,0.128936,0.045298,-0.044059,0.301420,0.124469,-0.107604,...,5.728443,-9.407360,-2.038582,1.370812,-5.477564,-0.336243,1.502100,1.546016,0.210300,2
1,514,C10-2,-0.448317,-0.227693,0.127213,-0.005069,0.028333,0.112086,0.028173,-0.080888,...,1.572523,-6.513134,-3.327943,5.796319,-3.204541,-0.909036,3.659749,-3.007935,1.133443,2
2,1028,C10-2,-0.372906,-0.141308,0.181552,-0.005005,0.046912,0.081046,0.008974,-0.084817,...,2.079188,-8.914544,-0.971781,3.080546,-6.735629,-1.390779,2.282022,1.280657,-0.801634,2
3,1285,C10-2,-0.173261,-0.330922,0.239251,0.008114,-0.084119,0.230205,0.123472,-0.168975,...,3.528395,-4.063998,-3.547382,2.289299,-2.786486,0.148371,3.511132,-2.909451,0.473595,2
4,1542,C10-2,-0.477709,-0.282062,0.235665,-0.032700,-0.003505,0.119395,0.032428,-0.049148,...,-2.390701,-4.856587,-3.907551,6.513190,3.495226,-0.992177,3.060692,-1.848124,-4.039866,-1
